# Stage 5 — Tabular foundation models, modern MLPs and Optuna ★ best submission

**Question:** do model families that learn differently from gradient-boosted trees add information?

- **TabICL** — a pretrained tabular foundation model (in-context learning, no training loop).
- **TabM** and **RealMLP** — modern tabular MLPs from `pytabkit`, used with their tuned defaults.
- **CatBoost + Optuna** — 94 seeded, time-boxed trials with pruning; the final model is scored on fold splits the search never saw.
- Stage 4's predictions are reused from disk, not retrained.

| result | CV logloss | public LB | wrong rows |
|---|---|---|---|
| **prob_blend** | **0.27691** | **0.108467** | **1094** |

**Finding:** the largest genuine CV gain of the project (−0.00039 from the new families). TabICL (28%) and TabM (22%) took the biggest blend weights. RealMLP was weak and slow; TabPFN was skipped (license not yet accepted).

**Runs on Kaggle** (GPU T4 x2, Internet on for `pip install`). Inputs and the saved-prediction chain are described in the [README](../README.md#reproducing).

In [1]:
import os, sys, glob, time, inspect, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(name)
    return os.path.abspath(hits[0]) if hits else None

TRAIN_PATH, TEST_PATH = find_file('Train.csv'), find_file('Test.csv')
if not TRAIN_PATH:
    raise FileNotFoundError('Train.csv not found - attach the dataset via "Add Input".')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# ---- knobs ----------------------------------------------------------------
FAST = os.environ.get('FAST') == '1'            # quick smoke test
N_FOLDS = 5
RUN = dict(tabicl=True, tabpfn=True, realmlp=True, tabm=True, cb_optuna=True)
BUDGET_MIN = dict(tabicl=20, tabpfn=20, realmlp=20, tabm=20, cb_optuna=25)  # per model, after tuning
OPTUNA_MINUTES = 3 if FAST else 35              # time box for the Optuna search
NEW_SEEDS = [0] if FAST else [0, 1, 2]          # fold splits for the new models (same splits as v4)
OPTUNA_SEEDS = [0] if FAST else [5, 6, 7, 8, 9] # final tuned CatBoost uses splits Optuna never scored
FULL_REFIT, FULL_WEIGHT = True, 0.5             # same test-prediction recipe as v4
# ---------------------------------------------------------------------------

def pip_install(*pkgs):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], capture_output=True, text=True)
    print('pip install', *pkgs, '->', 'ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-800:])
    return r.returncode == 0

for pkg in ['tabicl', 'pytabkit', 'optuna'] + (['tabpfn'] if RUN['tabpfn'] else []):
    pip_install(pkg)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('train:', TRAIN_PATH, '| out:', OUT_DIR, '| device:', DEVICE, '| FAST:', FAST, '| CPUs:', os.cpu_count())

pip install tabicl -> ok
pip install pytabkit -> ok
pip install optuna -> ok
pip install tabpfn -> ok
train: /kaggle/input/datasets/jeffmogaka/datadef/Train.csv | out: /kaggle/working | device: cuda | FAST: False | CPUs: 4


## Data (same features as v4) plus two encodings for the new models

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = (train['bank_account'] == 'Yes').astype(int).values
test_ids = test['uniqueid'] + ' x ' + test['country']

CATS = ['country', 'location_type', 'cellphone_access', 'gender_of_respondent',
        'relationship_with_head', 'marital_status', 'education_level', 'job_type']
NUMS = ['household_size', 'age_of_respondent']

X = train[CATS + NUMS].reset_index(drop=True)
X_test = test[CATS + NUMS].reset_index(drop=True)
both = pd.concat([X, X_test], ignore_index=True)

# one-hot + numeric floats: for the foundation models (TabICL, TabPFN)
OH = pd.get_dummies(both[CATS], dtype=np.float32)
OH[NUMS] = both[NUMS].astype(np.float32)
Xoh, Xoh_test = OH.iloc[:len(X)].reset_index(drop=True), OH.iloc[len(X):].reset_index(drop=True)

# pandas categoricals + numeric: for pytabkit (RealMLP, TabM), which embeds categories itself
CAT_DF = both.copy()
for c in CATS:
    CAT_DF[c] = pd.Categorical(CAT_DF[c])
CAT_DF[NUMS] = CAT_DF[NUMS].astype(np.float32)
Xcat, Xcat_test = CAT_DF.iloc[:len(X)].reset_index(drop=True), CAT_DF.iloc[len(X):].reset_index(drop=True)

print(X.shape, '| one-hot width', Xoh.shape[1], '| positive rate %.4f' % y.mean())

(23524, 10) | one-hot width 39 | positive rate 0.1408


## Harness (same folds as v4, plus a time budget)
`fit(train_rows, seed)` returns `predict(rows)` (`rows=None` = test). After the first seed, a model only runs as many
further seeds as fit in its time budget. Any error skips the model instead of killing the run.

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

FOLDS = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y)) for s in range(10)}
ALL_ROWS = np.arange(len(y))
OOF, TEST_FOLD, TEST_FULL, SCORES, NEW_MODELS = {}, {}, {}, {}, []

def error_rate(p, t=0.5):
    return np.mean((p > t).astype(int) != y)

def run_model(name, fit, seeds, budget_min=None):
    t0, done = time.time(), []
    oof = np.zeros((len(seeds), len(y)))
    tst, full = np.zeros(len(X_test)), np.zeros(len(X_test))
    for k, s in enumerate(seeds):
        if done and budget_min and (time.time() - t0) / 60 * (len(done) + 1) / len(done) > budget_min:
            print(f'  {name}: time budget reached, stopping after {len(done)} seed(s)')
            break
        t_seed = np.zeros(len(X_test))
        for tr_idx, va_idx in FOLDS[s]:
            predict = fit(tr_idx, s)
            oof[k, va_idx] = predict(va_idx)
            t_seed += predict(None) / N_FOLDS
        tst += t_seed
        if FULL_REFIT:
            full += fit(ALL_ROWS, s)(None)
        done.append(k)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    n = len(done)
    oof, tst, full = oof[done], tst / n, (full / n if FULL_REFIT else tst / n)
    per_seed = [error_rate(o) for o in oof]
    OOF[name], TEST_FOLD[name], TEST_FULL[name] = oof.mean(0), tst, full
    SCORES[name] = dict(cv_error=np.mean(per_seed), logloss=np.mean([log_loss(y, o) for o in oof]),
                        logloss_avg=log_loss(y, OOF[name]), seeds=n, minutes=(time.time() - t0) / 60)
    print(f"{name:14s} logloss {SCORES[name]['logloss']:.5f} (seed-avg {SCORES[name]['logloss_avg']:.5f}) | "
          f"CV error {np.mean(per_seed):.5f} | {n} seed(s) | {SCORES[name]['minutes']:.1f} min")

def try_model(name, fit, seeds, budget_min=None):
    if not RUN.get(name, True):
        return
    try:
        run_model(name, fit, seeds, budget_min)
        NEW_MODELS.append(name)
    except Exception as e:
        print(f'!! {name} skipped: {type(e).__name__}: {str(e)[:300]}')

def make(cls, **kw):
    # Pass only the arguments this library version accepts, so API differences don't crash the run.
    params = inspect.signature(cls.__init__).parameters
    if not any(p.kind == p.VAR_KEYWORD for p in params.values()):
        kw = {k: v for k, v in kw.items() if k in params}
    return cls(**kw)

## v4's models (reused, not retrained)
If stage 4's saved predictions are attached, its 8 models are loaded. Otherwise the three that carried v4's blend are retrained.

In [4]:
oof4, full4, fold4 = (find_file(f) for f in ['oof_v4.csv', 'test_probabilities_v4_full.csv', 'test_probabilities_v4_fold.csv'])
if oof4 and full4 and fold4:
    o, tf, td = pd.read_csv(oof4), pd.read_csv(full4), pd.read_csv(fold4)
    assert (o['y'].values == y).all(), 'oof_v4.csv rows do not match Train.csv'
    assert (tf['unique_id'].values == test_ids.values).all(), 'v4 test rows do not match Test.csv'
    for c in o.columns.drop('y'):
        OOF[c], TEST_FULL[c], TEST_FOLD[c] = o[c].values, tf[c].values, td[c].values
        SCORES[c] = dict(cv_error=error_rate(OOF[c]), logloss=np.nan, logloss_avg=log_loss(y, OOF[c]), seeds=np.nan, minutes=0.0)
    print('loaded v4 models:', list(o.columns.drop('y')))
else:
    print('v4 output not attached - retraining cb_cpu_deep, logreg_cross and lightgbm (~25 min).')
    from catboost import CatBoostClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import OneHotEncoder
    import lightgbm as lgb
    SEEDS4 = [0] if FAST else [0, 1, 2, 3, 4]

    def cb_deep_fit(tr_idx, seed):
        m = CatBoostClassifier(iterations=300 if FAST else 1000, learning_rate=0.03, depth=7, l2_leaf_reg=10,
                               max_ctr_complexity=3, random_strength=1.5, boosting_type='Ordered', border_count=254,
                               thread_count=-1, random_seed=seed + 500, verbose=0, allow_writing_files=False)
        m.fit(X.iloc[tr_idx], y[tr_idx], cat_features=CATS)
        return lambda rows: m.predict_proba(X_test if rows is None else X.iloc[rows])[:, 1]

    def crosses(df):
        d = df[CATS].copy()
        d['age_b'] = pd.cut(df['age_of_respondent'], [0, 20, 25, 30, 35, 40, 50, 60, 70, 200]).astype(str)
        d['hh_b'] = df['household_size'].clip(1, 12).astype(str)
        base = list(d.columns)
        for i in range(len(base)):
            for j in range(i + 1, len(base)):
                d[base[i] + '*' + base[j]] = d[base[i]] + '|' + d[base[j]]
        for a, b, c in [('country', 'education_level', 'job_type'), ('country', 'location_type', 'cellphone_access'),
                        ('country', 'age_b', 'gender_of_respondent'), ('country', 'education_level', 'age_b'),
                        ('country', 'relationship_with_head', 'marital_status')]:
            d[f'{a}*{b}*{c}'] = d[a] + '|' + d[b] + '|' + d[c]
        return d
    Xx, Xx_test = crosses(X), crosses(X_test)

    def lr_fit(tr_idx, seed):
        enc = OneHotEncoder(handle_unknown='ignore', min_frequency=5)
        m = LogisticRegression(C=0.02, max_iter=3000).fit(enc.fit_transform(Xx.iloc[tr_idx]), y[tr_idx])
        return lambda rows: m.predict_proba(enc.transform(Xx_test if rows is None else Xx.iloc[rows]))[:, 1]

    def lgb_fit(tr_idx, seed):
        m = lgb.LGBMClassifier(n_estimators=150 if FAST else 700, learning_rate=0.02, num_leaves=24, min_child_samples=40,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.7, cat_smooth=20, cat_l2=10,
                               reg_lambda=3, verbose=-1, n_jobs=-1, random_state=seed)
        m.fit(Xcat.iloc[tr_idx], y[tr_idx])
        return lambda rows: m.predict_proba(Xcat_test if rows is None else Xcat.iloc[rows])[:, 1]

    run_model('cb_cpu_deep', cb_deep_fit, SEEDS4)
    run_model('logreg_cross', lr_fit, SEEDS4)
    run_model('lightgbm', lgb_fit, SEEDS4)
BASE_MODELS = list(OOF)

loaded v4 models: ['cb_cpu', 'cb_cpu_v2feat', 'cb_cpu_slow', 'cb_cpu_deep', 'lightgbm', 'lightgbm_xt', 'logreg_cross', 'embed_nn']


## New model 1: TabICL (tabular foundation model)
A transformer pretrained on millions of synthetic tables. It reads the training rows as context and predicts test rows in one forward pass: no training loop, no tuning.

In [5]:
try:
    from tabicl import TabICLClassifier
    def tabicl_fit(tr_idx, seed):
        m = make(TabICLClassifier, n_estimators=4 if FAST else 16, random_state=seed, device=DEVICE, verbose=False)
        m.fit(Xoh.iloc[tr_idx].values, y[tr_idx])
        return lambda rows: m.predict_proba(Xoh_test.values if rows is None else Xoh.iloc[rows].values)[:, 1]
    try_model('tabicl', tabicl_fit, NEW_SEEDS, BUDGET_MIN['tabicl'])
except ImportError as e:
    print('!! tabicl not available:', e)

tabicl-classifier-v2-20260212.ckpt: reconstructing file:   0%|          |  0.00B /  110MB            

tabicl-classifier-v2-20260212.ckpt: downloading bytes:           |  0.00B            

  tabicl: time budget reached, stopping after 2 seed(s)
tabicl         logloss 0.27887 (seed-avg 0.27844) | CV error 0.11152 | 2 seed(s) | 13.5 min


## New model 2 (optional): TabPFN
Runs only if a `TABPFN_TOKEN` Kaggle secret exists (free token from ux.priorlabs.ai; the weights are licensed for non-commercial/research use).

In [6]:
token = os.environ.get('TABPFN_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception:
        token = None
if RUN['tabpfn'] and token:
    os.environ['TABPFN_TOKEN'] = token
    try:
        from tabpfn import TabPFNClassifier
        def tabpfn_fit(tr_idx, seed):
            m = make(TabPFNClassifier, n_estimators=2 if FAST else 8, random_state=seed, device=DEVICE,
                     ignore_pretraining_limits=True)
            m.fit(Xoh.iloc[tr_idx].values, y[tr_idx])
            return lambda rows: m.predict_proba(Xoh_test.values if rows is None else Xoh.iloc[rows].values)[:, 1]
        try_model('tabpfn', tabpfn_fit, NEW_SEEDS, BUDGET_MIN['tabpfn'])
    except ImportError as e:
        print('!! tabpfn not available:', e)
else:
    print('TabPFN skipped (no TABPFN_TOKEN secret).')

!! tabpfn skipped: TabPFNLicenseError: TabPFN requires a one-time license acceptance to download
model weights for local inference, but no interactive terminal
is available.

To authenticate in a non-interactive environment:
  1. Open https://ux.priorlabs.ai in a browser and log in (or register)
  2. Accept the license on the Licenses ta


## New models 3 and 4: RealMLP and TabM (pytabkit)
Both use the library's tuned defaults ("TD"/"D"), which were designed to work well without per-dataset tuning.
They hold out part of each training fold internally for early stopping, so the scoring fold stays unseen.

In [7]:
try:
    import pytabkit
    from pytabkit import RealMLP_TD_Classifier
    try:
        from pytabkit import TabM_D_Classifier
    except ImportError:
        TabM_D_Classifier = None
    def pytabkit_fit(cls):
        def fit(tr_idx, seed):
            m = make(cls, random_state=seed, device=DEVICE, verbosity=0, n_cv=1)
            m.fit(Xcat.iloc[tr_idx], y[tr_idx])
            return lambda rows: m.predict_proba(Xcat_test if rows is None else Xcat.iloc[rows])[:, 1]
        return fit
    try_model('realmlp', pytabkit_fit(RealMLP_TD_Classifier), NEW_SEEDS, BUDGET_MIN['realmlp'])
    if TabM_D_Classifier is not None:
        try_model('tabm', pytabkit_fit(TabM_D_Classifier), NEW_SEEDS, BUDGET_MIN['tabm'])
    else:
        print('!! TabM_D_Classifier not in this pytabkit version')
except ImportError as e:
    print('!! pytabkit not available:', e)

  realmlp: time budget reached, stopping after 1 seed(s)
realmlp        logloss 0.29366 (seed-avg 0.29366) | CV error 0.11342 | 1 seed(s) | 12.4 min
tabm           logloss 0.28031 (seed-avg 0.27832) | CV error 0.11258 | 3 seed(s) | 6.2 min


## New model 5: CatBoost (CPU, ordered boosting) tuned by Optuna
- Search starts from v4's best model (`cb_cpu_deep`), so the tuned result can only match or beat it on the tuning split.
- Objective: 5-fold logloss on fold split 0, with pruning of clearly bad trials after each fold. Seeded, time-boxed.
- The final model is then trained on fold splits 5–9, which Optuna never scored, so its CV score is not flattered by the search.

In [8]:
from catboost import CatBoostClassifier
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def cb_params(trial):
        lr = trial.suggest_float('learning_rate', 0.015, 0.08, log=True)
        return dict(learning_rate=lr, iterations=int(round(30 / lr)),   # keeps lr x iterations ~ v4's 0.03 x 1000
                    depth=trial.suggest_int('depth', 4, 8),
                    l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 1, 40, log=True),
                    random_strength=trial.suggest_float('random_strength', 0.0, 4.0),
                    bagging_temperature=trial.suggest_float('bagging_temperature', 0.0, 2.0),
                    max_ctr_complexity=trial.suggest_int('max_ctr_complexity', 1, 4),
                    one_hot_max_size=trial.suggest_categorical('one_hot_max_size', [2, 4, 10]))

    def cb_model(p, seed):
        p = dict(p, iterations=300 if FAST else p['iterations'])
        return CatBoostClassifier(**p, boosting_type='Ordered', border_count=254, thread_count=-1,
                                  random_seed=seed, verbose=0, allow_writing_files=False)

    def objective(trial):
        p, oof = cb_params(trial), np.zeros(len(y))
        seen = []
        for f, (tr_idx, va_idx) in enumerate(FOLDS[0]):
            m = cb_model(p, 1000 + f).fit(X.iloc[tr_idx], y[tr_idx], cat_features=CATS)
            oof[va_idx] = m.predict_proba(X.iloc[va_idx])[:, 1]
            seen.extend(va_idx)
            trial.report(log_loss(y[seen], oof[seen]), f)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return log_loss(y, oof)

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=1))
    study.enqueue_trial(dict(learning_rate=0.03, depth=7, l2_leaf_reg=10, random_strength=1.5,
                             bagging_temperature=1.0, max_ctr_complexity=3, one_hot_max_size=2))
    t0 = time.time()
    study.optimize(objective, timeout=OPTUNA_MINUTES * 60, n_trials=2 if FAST else None)
    print(f'Optuna: {len(study.trials)} trials in {(time.time() - t0) / 60:.1f} min | '
          f'v4 settings {study.trials[0].value:.5f} -> best {study.best_value:.5f}')
    print('best params:', study.best_params)
    display(study.trials_dataframe()[['number', 'value', 'state']].sort_values('value').head(8))

    BEST_CB = cb_params(optuna.trial.FixedTrial(study.best_params))
    def cb_optuna_fit(tr_idx, seed):
        m = cb_model(BEST_CB, seed + 700).fit(X.iloc[tr_idx], y[tr_idx], cat_features=CATS)
        return lambda rows: m.predict_proba(X_test if rows is None else X.iloc[rows])[:, 1]
    try_model('cb_optuna', cb_optuna_fit, OPTUNA_SEEDS, BUDGET_MIN['cb_optuna'])
except ImportError as e:
    print('!! optuna not available:', e)

Optuna: 94 trials in 35.1 min | v4 settings 0.27933 -> best 0.27826
best params: {'learning_rate': 0.05851668697255306, 'depth': 4, 'l2_leaf_reg': 39.7308754409128, 'random_strength': 3.427150133418205, 'bagging_temperature': 0.8560638433330235, 'max_ctr_complexity': 2, 'one_hot_max_size': 10}


,number,value,state
5,5,0.277461,PRUNED
49,49,0.278261,COMPLETE
45,45,0.278329,COMPLETE
78,78,0.278376,PRUNED
24,24,0.278392,COMPLETE
69,69,0.278397,COMPLETE
34,34,0.278425,COMPLETE
46,46,0.278450,COMPLETE


cb_optuna      logloss 0.27862 (seed-avg 0.27774) | CV error 0.11189 | 5 seed(s) | 2.8 min


## Save the new models' predictions

In [9]:
pd.DataFrame({'y': y, **{k: OOF[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/oof_v5.csv', index=False)
pd.DataFrame({'unique_id': test_ids, **{k: TEST_FOLD[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/test_probabilities_v5_fold.csv', index=False)
pd.DataFrame({'unique_id': test_ids, **{k: TEST_FULL[k] for k in NEW_MODELS}}).to_csv(f'{OUT_DIR}/test_probabilities_v5_full.csv', index=False)
print('new models:', NEW_MODELS)

# How different is each new model from v4's best model? (lower correlation = more for the blend to use)
ref = 'cb_cpu_deep' if 'cb_cpu_deep' in OOF else BASE_MODELS[0]
for k in NEW_MODELS:
    print(f'{k:12s} correlation with {ref}: {np.corrcoef(OOF[k], OOF[ref])[0, 1]:.4f}')

new models: ['tabicl', 'realmlp', 'tabm', 'cb_optuna']
tabicl       correlation with cb_cpu_deep: 0.9911
realmlp      correlation with cb_cpu_deep: 0.9729
tabm         correlation with cb_cpu_deep: 0.9866
cb_optuna    correlation with cb_cpu_deep: 0.9940


## Combiners, judged on held-out logloss
Same as v4 (SLSQP probability blend and logit blend), plus a check of how much the **new** models add over v4's models alone.

In [10]:
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression

def logit(p):
    p = np.clip(p, 1e-5, 1 - 1e-5)
    return np.log(p / (1 - p))

class ProbBlend:                       # weighted average of probabilities, weights >= 0 summing to 1
    def fit(self, F, t):
        F = np.clip(F, 1e-6, 1 - 1e-6)
        def loss(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p))
        def grad(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -F.T @ (t / p - (1 - t) / (1 - p)) / len(t)
        k = F.shape[1]
        r = minimize(loss, np.ones(k) / k, jac=grad, method='SLSQP', bounds=[(0, 1)] * k,
                     constraints=[dict(type='eq', fun=lambda w: w.sum() - 1, jac=lambda w: np.ones(k))],
                     options=dict(maxiter=500, ftol=1e-10))
        self.w = np.clip(r.x, 0, None) / np.clip(r.x, 0, None).sum()
        return self
    def predict(self, F):
        return F @ self.w

class LogitBlend:
    def fit(self, F, t):
        self.m = LogisticRegression(C=1.0, max_iter=5000).fit(logit(F), t); return self
    def predict(self, F):
        return self.m.predict_proba(logit(F))[:, 1]

CANDIDATES = {'prob_blend': ProbBlend, 'logit_blend': LogitBlend}
META_SEEDS = [2024, 7, 99]

def held_out(make, cols):
    F, out = np.column_stack([OOF[n] for n in cols]), np.zeros(len(y))
    for ms in META_SEEDS:
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=ms).split(F, y):
            out[va] += make().fit(F[tr], y[tr]).predict(F[va]) / len(META_SEEDS)
    return out

names = list(OOF)
HELD_OOF = {k: held_out(make, names) for k, make in CANDIDATES.items()}
base_only = log_loss(y, held_out(ProbBlend, BASE_MODELS))
for k in CANDIDATES:
    print(f'{k:12s} held-out logloss {log_loss(y, HELD_OOF[k]):.5f} | error {error_rate(HELD_OOF[k]):.5f}')
print(f'prob_blend with v4 models only: {base_only:.5f}  -> new models change it by '
      f'{log_loss(y, HELD_OOF["prob_blend"]) - base_only:+.5f} (negative = better)')

P = np.column_stack([OOF[n] for n in names])
T_MIX = np.column_stack([(1 - FULL_WEIGHT) * TEST_FOLD[n] + FULL_WEIGHT * TEST_FULL[n] for n in names])
FITTED = {k: make().fit(P, y) for k, make in CANDIDATES.items()}
ranked = sorted(CANDIDATES, key=lambda k: log_loss(y, HELD_OOF[k]))
print('\nranking:', ranked)
print('prob_blend weights:', {n: round(float(w), 3) for n, w in zip(names, FITTED['prob_blend'].w) if w > 0.0005})

prob_blend   held-out logloss 0.27691 | error 0.11176
logit_blend  held-out logloss 0.27709 | error 0.11176
prob_blend with v4 models only: 0.27729  -> new models change it by -0.00039 (negative = better)

ranking: ['prob_blend', 'logit_blend']
prob_blend weights: {'cb_cpu_deep': 0.223, 'lightgbm': 0.115, 'logreg_cross': 0.031, 'tabicl': 0.277, 'realmlp': 0.003, 'tabm': 0.221, 'cb_optuna': 0.13}


## Cutoff check (noise floor 0.001, as before)

In [11]:
grid = np.round(np.arange(0.40, 0.601, 0.01), 2)
THRESH = {}
for k in CANDIDATES:
    o = HELD_OOF[k]
    errs = np.array([error_rate(o, t) for t in grid])
    THRESH[k] = grid[errs.argmin()] if errs.min() < error_rate(o) - 0.001 else 0.5
    print(f'{k:12s} best cutoff {grid[errs.argmin()]} (err {errs.min():.5f}) -> using {THRESH[k]}')

prob_blend   best cutoff 0.49 (err 0.11146) -> using 0.5
logit_blend  best cutoff 0.52 (err 0.11155) -> using 0.5


## Board (sorted by logloss)

In [12]:
board = pd.DataFrame(SCORES).T[['logloss', 'logloss_avg', 'cv_error', 'seeds', 'minutes']]
board['new'] = ['NEW' if k in NEW_MODELS else '' for k in board.index]
for k in CANDIDATES:
    board.loc['>> ' + k] = [np.nan, log_loss(y, HELD_OOF[k]), error_rate(HELD_OOF[k], THRESH[k]), np.nan, np.nan, '']
display(board.sort_values('logloss_avg').round(5))
print('Reference held-out logloss: v2 0.27754 (LB 1097 rows), v4 0.27729 (LB 1100 rows).')

,logloss,logloss_avg,cv_error,seeds,minutes,new
>> prob_blend,NaN,0.27691,0.11176,NaN,NaN,
>> logit_blend,NaN,0.27709,0.11176,NaN,NaN,
cb_cpu_deep,NaN,0.27764,0.11206,NaN,0.00000,
cb_optuna,0.27862,0.27774,0.11189,5.0,2.78269,NEW
cb_cpu_slow,NaN,0.27781,0.11172,NaN,0.00000,
cb_cpu,NaN,0.27798,0.11167,NaN,0.00000,
cb_cpu_v2feat,NaN,0.27828,0.11201,NaN,0.00000,
tabm,0.28031,0.27832,0.11258,3.0,6.24426,NEW
tabicl,0.27887,0.27844,0.11152,2.0,13.51198,NEW
logreg_cross,NaN,0.27957,0.11206,NaN,0.00000,


Reference held-out logloss: v2 0.27754 (LB 1097 rows), v4 0.27729 (LB 1100 rows).


## Write submissions
- `submission_v5_best.csv`: best combiner over v4 + new models. (primary submission)
- `submission_v5_second.csv`: runner-up combiner, as a spare.

In [13]:
def save(name, probs, t=0.5):
    sub = pd.DataFrame({'unique_id': test_ids, 'bank_account': (probs > t).astype(int)})
    path = f'{OUT_DIR}/submission_v5_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'{path}: {sub.bank_account.sum()} ones / {len(sub)} rows')
    return sub

best, second = ranked[0], ranked[1]
print('best =', best, '| second =', second)
SUBS = {'best': save('best', FITTED[best].predict(T_MIX), THRESH[best]),
        'second': save('second', FITTED[second].predict(T_MIX), THRESH[second])}

def labels(df):
    return df.set_index('unique_id')['bank_account'].reindex(test_ids).values

for prev_name in ['submission_v2_recommended.csv', 'submission_v4_best.csv']:
    prev_path = find_file(prev_name)
    if prev_path:
        prev = labels(pd.read_csv(prev_path))
        for k, sub in SUBS.items():
            print(f'{k:7s} vs {prev_name}: {(labels(sub) != prev).sum()} rows differ')

best = prob_blend | second = logit_blend
/kaggle/working/submission_v5_best.csv: 724 ones / 10086 rows
/kaggle/working/submission_v5_second.csv: 739 ones / 10086 rows
best    vs submission_v4_best.csv: 54 rows differ
second  vs submission_v4_best.csv: 51 rows differ
